In [1]:
import sympy as sp

In [2]:
# define independent variables 
x,y = sp.symbols('x, y', real=True)
xv = sp.Matrix([x,y])

# define the rhs of the governing equation
w = sp.symbols("omega")
u1,u2 = sp.symbols("u_x, u_y", real=True)
u1 = sp.Function("u_x")(x, y)
u2 = sp.Function("u_y")(x, y)
u = sp.Matrix([u1,u2])
grad_w = sp.Matrix([sp.Derivative(w,x), sp.Derivative(w,y)])
FU = -u.dot(grad_w)
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [3]:
# define q(t)
N = 2 #number of vortexes

q = sp.Matrix()

A = sp.symbols("A", real=True)
L = sp.symbols("L", real=True, positive=True)

# xc= sp.symbols("x_c", real=True)
# yc= sp.symbols("y_c", real=True)

xc= sp.Matrix()
yc= sp.Matrix()
# r = sp.Matrix()

for i in range(N):
    xc = sp.Matrix([xc, sp.symbols("x_c_"+str(i+1), real=True)])
    yc = sp.Matrix([yc, sp.symbols("y_c_"+str(i+1), real=True)])
    # r = sp.Matrix([r, sp.symbols("r_"+str(i+1), real=True, positive=True)])
    # r = sp.Matrix([r, sp.Function("r_"+str(i+1))(x, y, xc[i], yc[i])])

q = sp.Matrix([A, L, xc, yc])
# qr = sp.Matrix([A, L, r])

q

Matrix([
[    A],
[    L],
[x_c_1],
[x_c_2],
[y_c_1],
[y_c_2]])

In [4]:
# define the ansatz u_hat(x; q)
ansatz_gamma = 0
for i in range(N):
    ansatz_gamma = ansatz_gamma + A*sp.exp(-(((x-xc[i])**2+(y-yc[i])**2))/L**2)
# ansatz_gamma = A*sp.exp(-(((x-xc)**2+(y-yc)**2))/L**2) + A*sp.exp(-(((x+xc)**2+(y+yc)**2))/L**2)

ansatz_gamma

A*exp((-(x - x_c_1)**2 - (y - y_c_1)**2)/L**2) + A*exp((-(x - x_c_2)**2 - (y - y_c_2)**2)/L**2)

In [5]:
ansatz_u = sp.Matrix([
    sp.Derivative(ansatz_gamma,y).doit().simplify(),
    -sp.Derivative(ansatz_gamma, x).doit().simplify()
])

ansatz_u

Matrix([
[ 2*A*((-y + y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-y + y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**2],
[-2*A*((-x + x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-x + x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**2]])

In [6]:
ansatz = (- sp.Derivative(ansatz_gamma, x, 2) - sp.Derivative(ansatz_gamma, y, 2)).doit()
ansatz.simplify()

4*A*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4

In [7]:
# compute partial derivatives du/dqi
dwdq = ansatz.diff(q)

dwdq.simplify()

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         4*(L**2*(exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2)) - (x - x_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (x - x_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) - (y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) - (y - y_c_2)**2*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))/L**4],
[-8*A*exp(-(x - x_c_1)**2/L**2 - (y - y_c_1)**2/L**2)/L**3 - 8*A*exp(-(x - x_c_2)**2/L**2 - (y 

In [8]:
# data
a = 1
l = 1
xc1 = 1
yc1 = 0
xc2 = -1
yc2 = 0

def sub_data(f):
    return f.subs(A,a).subs(L,l).subs(xc[0],xc1).subs(yc[0],yc1).subs(xc[1],xc2).subs(yc[1],yc2)

In [20]:
xmin,xmax = sp.symbols("x_{min}, x_{max}")
# define the inner product according to the problem
def inner_prod_H(f, g):
    ix = sp.integrate((sub_data(f*g)).expand(),(x, -sp.oo, sp.oo))
    return sp.integrate(ix.expand(),(y, -sp.oo, sp.oo)).expand()

In [21]:
inner_prod_H(dwdq[0], dwdq[0]).simplify()
# m00 = (dwdq[0]**2).simplify()

# m00i = sp.integrate(m00.expand(), (x, -sp.oo, +sp.oo))

4*pi*(-2 + (-erf(sqrt(2)) - erfc(sqrt(2)) + 3)*exp(2))*exp(-2)

In [11]:
# construct the matrix M_ij = <du/dqi, du/dqj>_H
n = len(q)
M = sp.zeros(n, n)

for i in range(n):
    M[i, i] = inner_prod_H(dwdq[i], dwdq[i]).simplify()
    print(M[i, i])
    for j in range(i+1, n):
        M[i, j] = inner_prod_H(dwdq[i], dwdq[j]).simplify()
        M[j,i] = M[i,j]
        print(M[i, j])


4*pi*(-2 + (-erf(sqrt(2)) - erfc(sqrt(2)) + 3)*exp(2))*exp(-2)
4*pi*((-3 + erfc(sqrt(2)) + erf(sqrt(2)))*exp(2) - 2)*exp(-2)
8*pi*exp(-2)
-8*pi*exp(-2)
0
0
16*pi*(2 + (-erf(sqrt(2)) - erfc(sqrt(2)) + 3)*exp(2))*exp(-2)
-16*pi*exp(-2)
16*pi*exp(-2)
0
0
6*pi*(-erf(sqrt(2)) - erfc(sqrt(2)) + 3)
-4*pi*exp(-2)
0
0
6*pi*(-erf(sqrt(2)) - erfc(sqrt(2)) + 3)
0
0
6*pi*(-erf(sqrt(2)) - erfc(sqrt(2)) + 3)
-4*pi*exp(-2)
6*pi*(-erf(sqrt(2)) - erfc(sqrt(2)) + 3)


In [12]:
M

Matrix([
[4*pi*(-2 + (-erf(sqrt(2)) - erfc(sqrt(2)) + 3)*exp(2))*exp(-2),  4*pi*((-3 + erfc(sqrt(2)) + erf(sqrt(2)))*exp(2) - 2)*exp(-2),                             8*pi*exp(-2),                            -8*pi*exp(-2),                                        0,                                        0],
[ 4*pi*((-3 + erfc(sqrt(2)) + erf(sqrt(2)))*exp(2) - 2)*exp(-2), 16*pi*(2 + (-erf(sqrt(2)) - erfc(sqrt(2)) + 3)*exp(2))*exp(-2),                           -16*pi*exp(-2),                            16*pi*exp(-2),                                        0,                                        0],
[                                                  8*pi*exp(-2),                                                 -16*pi*exp(-2), 6*pi*(-erf(sqrt(2)) - erfc(sqrt(2)) + 3),                            -4*pi*exp(-2),                                        0,                                        0],
[                                                 -8*pi*exp(-2),                                 

In [13]:
FU

-u_x(x, y)*Derivative(omega, x) - u_y(x, y)*Derivative(omega, y)

In [14]:
# compute rhs from the ansatz
Fua = FU.subs(u1, ansatz_u[0]).subs(u2, ansatz_u[1]).subs(w, ansatz).doit()
Fua.simplify()

16*A**2*(((x - x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))*(2*L**2*((y - y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (y - y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2)) + (x - x_c_1)**2*(-y + y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)**2*(-y + y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + (-y + y_c_1)**3*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-y + y_c_2)**3*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2)) - ((y - y_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (y - y_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2))*(2*L**2*((x - x_c_1)*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (x - x_c_2)*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2)) + (-x + x_c_1)**3*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-x + x_c_1)*(y - y_c_1)**2*exp(-((x - x_c_1)**2 + (y - y_c_1)**2)/L**2) + (-x + x_c_2)**3*exp(-((x - x_c_2)**2 + (y - y_c_2)**2)/L**2) + (-x + x_c_2)*(y - y_c

In [15]:
# compute f
n = len(q)
f = sp.zeros(n, 1)

for i in range(n):
    f[i] = inner_prod_H(dwdq[i], Fua).simplify()
    print(f[i])

f

0
0
0
0
7168*pi*exp(-8/3)/243
-7168*pi*exp(-8/3)/243


Matrix([
[                     0],
[                     0],
[                     0],
[                     0],
[ 7168*pi*exp(-8/3)/243],
[-7168*pi*exp(-8/3)/243]])

In [16]:
q_dot = M.inv()*f

q_dot.simplify()

In [19]:
q_dot

Matrix([
[                                                                                                                                                                                                                                         0],
[                                                                                                                                                                                                                                         0],
[                                                                                                                                                                                                                                         0],
[                                                                                                                                                                                                                                         0],
[3584*(-2 + 3*(-erf(sqrt(2)) - erfc(sqr